In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import math, time
import datetime

# 嘗試匯入 yfinance，如果沒有安裝會提示
try:
    import yfinance as yf
except ImportError:
    print("請先安裝 yfinance: pip install yfinance")

# 1. 定義 LSTM 模型 (源自您的 Notebook)
class LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(LSTM, self).__init__()
        # Hidden dimensions
        self.hidden_dim = hidden_dim
        # Number of hidden layers
        self.num_layers = num_layers

        # batch_first=True causes input/output tensors to be of shape
        # (batch_dim, seq_dim, feature_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_()
        # Initialize cell state
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).requires_grad_()

        # We need to detach as we are doing truncated backpropagation through time (BPTT)
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        # Index hidden state of last time step
        out = self.fc(out[:, -1, :]) 
        return out

# 2. 資料處理函數 (改寫自 Notebook，適配 yfinance 資料格式)
def load_data(stock, look_back):
    data_raw = stock.values # convert to numpy array
    data = []
    
    # create all possible sequences of length look_back
    for index in range(len(data_raw) - look_back): 
        data.append(data_raw[index: index + look_back])
    
    data = np.array(data);
    test_set_size = int(np.round(0.2*data.shape[0]));
    train_set_size = data.shape[0] - (test_set_size);
    
    x_train = data[:train_set_size,:-1,:]
    y_train = data[:train_set_size,-1,:]
    
    x_test = data[train_set_size:,:-1]
    y_test = data[train_set_size:,-1,:]
    
    return [x_train, y_train, x_test, y_test]

# 3. 主執行邏輯：針對三檔股票進行預測
def predict_stock(ticker, stock_name):
    print(f"\n正在處理: {stock_name} ({ticker}) ...")
    
    # 下載資料 (過去 5 年)
    end_date = datetime.datetime.now()
    start_date = end_date - datetime.timedelta(days=5*365)
    
    try:
        df = yf.download(ticker, start=start_date, end=end_date)
        if df.empty:
            print(f"無法下載 {ticker} 的資料。")
            return
    except Exception as e:
        print(f"下載失敗: {e}")
        return

    # 只取 'Close' 欄位並處理缺失值
    df = df[['Close']]
    df = df.fillna(method='ffill')

    # 正規化 (Normalization)
    scaler = MinMaxScaler(feature_range=(-1, 1))
    df['Close'] = scaler.fit_transform(df['Close'].values.reshape(-1,1))

    # 準備訓練資料
    look_back = 60 # Notebook 中設定的數值
    x_train, y_train, x_test, y_test = load_data(df, look_back)

    # 轉為 PyTorch Tensors
    x_train = torch.from_numpy(x_train).type(torch.Tensor)
    x_test = torch.from_numpy(x_test).type(torch.Tensor)
    y_train = torch.from_numpy(y_train).type(torch.Tensor)
    y_test = torch.from_numpy(y_test).type(torch.Tensor)

    # 設定模型參數 (依照 Notebook 設定)
    input_dim = 1
    hidden_dim = 32
    num_layers = 2 
    output_dim = 1
    num_epochs = 100

    model = LSTM(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers)
    loss_fn = torch.nn.MSELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=0.01)

    # 訓練模型
    print("開始訓練...")
    hist = np.zeros(num_epochs)
    for t in range(num_epochs):
        y_train_pred = model(x_train)
        loss = loss_fn(y_train_pred, y_train)
        
        if t % 20 == 0:
            print(f"Epoch {t} MSE: {loss.item()}")
        hist[t] = loss.item()

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

    # 進行預測
    y_test_pred = model(x_test)

    # 反向轉換 (Inverse Transform) 取得真實股價
    y_train_pred = scaler.inverse_transform(y_train_pred.detach().numpy())
    y_train_real = scaler.inverse_transform(y_train.detach().numpy())
    y_test_pred = scaler.inverse_transform(y_test_pred.detach().numpy())
    y_test_real = scaler.inverse_transform(y_test.detach().numpy())

    # 繪圖
    plt.figure(figsize=(12, 5))
    
    # 繪製歷史數據與測試數據的比較
    # 這裡我們主要關注測試集 (最近的數據)
    # 建立對應的時間索引
    test_dates = df.index[len(df) - len(y_test_real):]
    
    plt.plot(test_dates, y_test_real, color='red', label=f'Real {stock_name} Price')
    plt.plot(test_dates, y_test_pred, color='blue', label=f'Predicted {stock_name} Price')
    
    plt.title(f'{stock_name} Price Prediction (LSTM)')
    plt.xlabel('Date')
    plt.ylabel('Price (TWD)')
    plt.legend()
    plt.grid(True)
    plt.show()

# 4. 定義股票清單並執行
stocks = [
    ('2330.TW', 'TSMC (台積電)'),
    ('2454.TW', 'MediaTek (聯發科)'),
    ('2308.TW', 'Delta (台達電)')
]

# 需要 matplotlib 顯示中文可能需要額外設定字型，這裡主要使用英文標籤以避免亂碼
for ticker, name in stocks:
    predict_stock(ticker, name)

ModuleNotFoundError: No module named 'pandas'

In [3]:
conda install -c conda-forge yfinance

^C

Note: you may need to restart the kernel to use updated packages.
